# Phase 1 — Data Preparation
Load, merge, and aggregate all CSV sources into one row per customer.

In [25]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = "../data/raw/"

## 1. Load Data

### Customers

In [17]:
customer_cols = ['cus_code', 'group_name__c', 'cus_name', 'Segment__c', 'Segment_Group__c', 'NumberOfEmployees', 'JobType', 'SalesPerson']
customers = pd.read_csv(os.path.join(DATA_DIR, "Customers.csv"), usecols=customer_cols)
print(f"Customers: {customers.shape[0]:,} rows — {customers.shape[1]} columns")
customers.head()

Customers: 23,031 rows — 8 columns


,group_name__c,cus_code,cus_name,Segment__c,Segment_Group__c,NumberOfEmployees,JobType,SalesPerson
0,Regus,38783,REGIONAL BUSINESS CENTRES GREECE NO1 ΕΠΕ,Inside Sales,SME,2.0,Other,Aggeliki Gatzia
1,Bluepoint Group,39932,FANCY DRESS LTD,Field Sales - South Greece,MM,73.0,Apparel,Ioanna Kampouridou
2,Dhl,30034,DHL EXPRESS ΕΛΛΑΣ ΑΝΩΝΥΜΗ ΕΤΑΙΡΕΙΑ ΤΑΧΥΜΕΤΑΦΟΡΩΝ,Key Accounts - South Greece,LNA,475.0,Transportation & Logistics,Dimitrios Segredos
3,Pirelli Group,32042,ΕΛΑΣΤΙΚΑ PIRELLI Α.Ε.Ε.,Low Farming,MM,29.0,Wholesale Trade,Xenofon Petrocheilos
4,Geo Sol Group,52939,ΘΕΟΔΩΡΙΔΟΥ ΜΑΡΙΑ ΘΕΟΔΩΡΑ ΔΗΜΗΤΡΙΟΣ,Field Sales - South Greece,MM,15.0,Other,Konstantinos Miaoulis


### Loads

In [18]:
loads = pd.read_csv(
    os.path.join(DATA_DIR, "Loads.csv"),
    usecols=['cus_code', 'ode_net_amt', 'vou_label', 'vou_code', 'ord_posting_date']
)
loads['ord_posting_date'] = pd.to_datetime(loads['ord_posting_date'], utc=True)
loads = loads[loads['ord_posting_date'] >= '2023-01-01'].reset_index(drop=True)
print(f"Loads (2023+): {loads.shape[0]:,} rows — {loads.shape[1]} columns")
loads.head()

Loads (2023+): 10,025,945 rows — 5 columns


,cus_code,ode_net_amt,vou_label,vou_code,ord_posting_date
0,31878,400.0,Spendeo,13,2023-01-09 00:00:00+00:00
1,39146,100.0,Ticket Restaurant Card,19,2023-01-02 00:00:00+00:00
2,30813,130.0,Ticket Restaurant Card,19,2023-01-10 00:00:00+00:00
3,45107,115.0,Ticket Restaurant Card,19,2023-01-12 00:00:00+00:00
4,33986,103.5,Ticket Restaurant Card,19,2023-01-15 00:00:00+00:00


### Product

In [19]:
product = pd.read_csv(os.path.join(DATA_DIR, "Product.csv"))
print(f"Product: {product.shape[0]:,} rows — {product.shape[1]} columns")
product

Product: 16 rows — 3 columns


,vou_code,vou_label,ProductFamily
0,1,Ticket Restaurant,EB
1,2,Ticket Compliments,I&R
2,3,Ticket Car,I&R
3,4,Ticket Service,PSP
4,13,Spendeo,F&F
5,14,Ticket Car Fuel Card,F&F
6,17,<Unused For Future Comp Card>,NaN
7,18,Ticket Compliments Gift Card,I&R
8,19,Ticket Restaurant Card,EB
9,20,Ticket Restaurant Lite,TRL


## 2. Enrich with Dimensions

In [20]:
loads_enriched = (
    loads
    .merge(product[['vou_code', 'ProductFamily']], on='vou_code', how='left')
    .merge(customers, on='cus_code', how='left')
)

print(f"Enriched loads: {loads_enriched.shape[0]:,} rows — {loads_enriched.shape[1]} columns")
loads_enriched.head()

Enriched loads: 10,025,945 rows — 13 columns


,cus_code,ode_net_amt,vou_label,vou_code,ord_posting_date,ProductFamily,group_name__c,cus_name,Segment__c,Segment_Group__c,NumberOfEmployees,JobType,SalesPerson
0,31878,400.0,Spendeo,13,2023-01-09 00:00:00+00:00,F&F,NaN,ZZ DOT ΔΙΑΦΗΜΙΣΤΙΚΗ ΕΜΠΟΡΙΚΗ ΚΑΙ ΣΥΜΒΟΥΛΕΥΤΙΚΗ...,Field Sales - South Greece,MM,29.0,Marketing and advertising agencies,Ilias ROUSELIS
1,39146,100.0,Ticket Restaurant Card,19,2023-01-02 00:00:00+00:00,EB,Samsung Group,SAMSUNG SDS EUROPE LIMITED ΥΠΟΚΑΤΑΣΤΗΜΑ ΕΛΛΑΔΟΣ,Field Sales - South Greece,MM,5.0,Transportation & Logistics,Evgenia Takaki
2,30813,130.0,Ticket Restaurant Card,19,2023-01-10 00:00:00+00:00,EB,Vioiatriki Group,ΒΙΟΪΑΤΡΙΚΗ ΙΔ ΠΟΛΥΙΑΤΡΕΙΟ – ΙΔ ΔΙΑΓΝΩΣΤΙΚΟ ΕΡΓ...,Key Accounts - South Greece,LNA,2276.0,Healthcare,Mary Zoura
3,45107,115.0,Ticket Restaurant Card,19,2023-01-12 00:00:00+00:00,EB,NaN,G. F. ΤΕΧΝΙΚΗ ΙΚΕ,Inside Sales,SME,23.0,Other,Konstantina Stamouli
4,33986,103.5,Ticket Restaurant Card,19,2023-01-15 00:00:00+00:00,EB,Dei Group,ΔΙΑΧΕΙΡΙΣΤΗΣ ΕΛΛΗΝΙΚΟΥ ΔΙΚΤΥΟΥ ΗΛΕΚΤΡΙΚΗΣ ΕΝΕΡ...,Public Sector,PC,6297.0,NaN,Thomas Alexandrou


## 3. Monthly Aggregation per Customer & Product

In [26]:
loads_enriched['month'] = loads_enriched['ord_posting_date'].dt.to_period('M')

groupby_cols = [
    'cus_code', 'group_name__c', 'cus_name',
    'Segment__c', 'NumberOfEmployees', 'JobType', 'SalesPerson',
    'ProductFamily',
    'month',
]

loads_monthly = (
    loads_enriched
    .groupby(groupby_cols, as_index=False)
    .agg(total_amt=('ode_net_amt', 'sum'), order_count=('ode_net_amt', 'count'))
    .sort_values(['cus_code', 'month'])
    .reset_index(drop=True)
)

print(f"Monthly aggregated: {loads_monthly.shape[0]:,} rows — {loads_monthly.shape[1]} columns")
loads_monthly.head()

Monthly aggregated: 89,799 rows — 11 columns


,cus_code,group_name__c,cus_name,Segment__c,NumberOfEmployees,JobType,SalesPerson,ProductFamily,month,total_amt,order_count
0,30028,Pierre Fabre,PIERRE FABRE ΕΛΛΑΣ ΑΕ,Field Sales - South Greece,131.0,Chemicals & Plastic,Ioanna Kampouridou,EB,2023-01,6893.00,80
1,30028,Pierre Fabre,PIERRE FABRE ΕΛΛΑΣ ΑΕ,Field Sales - South Greece,131.0,Chemicals & Plastic,Ioanna Kampouridou,EB,2023-02,940.00,20
2,30028,Pierre Fabre,PIERRE FABRE ΕΛΛΑΣ ΑΕ,Field Sales - South Greece,131.0,Chemicals & Plastic,Ioanna Kampouridou,EB,2023-03,29223.00,174
3,30028,Pierre Fabre,PIERRE FABRE ΕΛΛΑΣ ΑΕ,Field Sales - South Greece,131.0,Chemicals & Plastic,Ioanna Kampouridou,EB,2023-04,10524.00,91
4,30028,Pierre Fabre,PIERRE FABRE ΕΛΛΑΣ ΑΕ,Field Sales - South Greece,131.0,Chemicals & Plastic,Ioanna Kampouridou,EB,2023-05,41102.03,112


In [28]:
loads_monthly

,cus_code,group_name__c,cus_name,Segment__c,NumberOfEmployees,JobType,SalesPerson,ProductFamily,month,total_amt,order_count
0,30028,Pierre Fabre,PIERRE FABRE ΕΛΛΑΣ ΑΕ,Field Sales - South Greece,131.0,Chemicals & Plastic,Ioanna Kampouridou,EB,2023-01,6893.00,80
1,30028,Pierre Fabre,PIERRE FABRE ΕΛΛΑΣ ΑΕ,Field Sales - South Greece,131.0,Chemicals & Plastic,Ioanna Kampouridou,EB,2023-02,940.00,20
2,30028,Pierre Fabre,PIERRE FABRE ΕΛΛΑΣ ΑΕ,Field Sales - South Greece,131.0,Chemicals & Plastic,Ioanna Kampouridou,EB,2023-03,29223.00,174
3,30028,Pierre Fabre,PIERRE FABRE ΕΛΛΑΣ ΑΕ,Field Sales - South Greece,131.0,Chemicals & Plastic,Ioanna Kampouridou,EB,2023-04,10524.00,91
4,30028,Pierre Fabre,PIERRE FABRE ΕΛΛΑΣ ΑΕ,Field Sales - South Greece,131.0,Chemicals & Plastic,Ioanna Kampouridou,EB,2023-05,41102.03,112
...,...,...,...,...,...,...,...,...,...,...,...
89794,54478,Sakarellos Group,DIAGNOSTICA ΙΔΙΩΤΙΚΑ ΔΙΑΓΝΩΣΤΙΚΑ ΕΡΓΑΣΤΗΡΙΑ ΙΑ...,Field Sales - South Greece,25.0,Healthcare,Evgenia Takaki,EB,2026-04,125.00,1
89795,54516,Mitrousis group,MITROUSIS ΣΥΜΒΟΥΛΕΥΤΙΚΗ ΜΑΚΕΔΟΝΙΑΣ ΘΡΑΚΗΣ ΜΟΝΟ...,Inside Sales,1.0,Consulting,Konstantina Stamouli,EB,2026-04,25.00,1
89796,54519,sushaki group,ΣΟΥΣΑΚΙ ΝΤΕΛΙΒΕΡΙ Ε Ε,Field Sales - North Greece,30.0,Retail,Vasilis Savvoglou,EB,2026-03,528.00,4
89797,54552,dr pharmacy group,DRP GROUP Ι.Κ.Ε.,Inside Sales,1.0,Other,Katerina Loukogeorgaki,EB,2026-04,523.20,5
